<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_DISCOVERY_GEOMOE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_DISCOVERY_GEOMOE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS discovery-first GeoMoE (code rev dgm.2-r1)

**One notebook, one CONFIG, run top to bottom.** `FAST_DIAGNOSTIC` measures candidate coverage first ([00]-[06]); `FULL` continues from the same cached artifacts into fusion, gating, deployment and export.

| | |
|---|---|
| Runtime | Colab **CPU**, High-RAM. No GPU, no neural network is trained. |
| Reference | **A_submitted_W2** = the frozen submitted W2 policy (public DTI 0.0942), one cell for every scenario |
| Outputs | `MyDrive/GEMS_2/outputs/discovery_geomoe/` |
| Code | `gems_bench/geomoe/` - docs `docs/geomoe/DISCOVERY_GEOMOE.md` |

Every stage checks its upstream stages: a failed / blocked dependency gives `BLOCKED_UPSTREAM` (compute) or `NOT_EVALUATED` (decision) and **no conclusion**. A failing test gives `TESTS_FAILED` and blocks candidates, modeling, selection and export. Cached results are reused only after validation (`REUSED_VALID_CACHE`).

**After a disconnect:** restart, run cells 1-2, then the stage cells again (finished units / stages are reused).

## 1 - Configuration (the only cell you edit)

`mode`: `FAST_DIAGNOSTIC` ([00]-[06] + integrity + reports) | `FULL` | `RESUME` (= FULL, cache forced). dgm.1 keys (`experts`, `experiments`, `export`) still work.

In [1]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                          # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'mode': 'FULL',              # FAST_DIAGNOSTIC | FULL | RESUME
    'reuse_cached': True,
    'train_neural_models': False,           # True is refused

    'backfill_missing_g02': False,          # True: compute ONLY the missing G02 unit rasters (frozen G02 config)
    'dem': 'cached_only',                   # cached_only | off (never a DEM download)

    'run_synthetic_tests': True,            # self-tests + pytest, ENFORCED
    'strict_dependency_checks': True,

    'export_validated_candidate': True,
    'allow_exploratory_export': False,

    'save_all_outputs': True,
    'sync_to_drive': True,
    'overrides': {},                        # optional overrides of configs/geomoe/discovery_geomoe.yaml
}
PRIVATE_REPO = False
EXPECT = {'code_rev': 'dgm.2-r1', 'commit': None}   # set 'commit' to the pushed SHA

## 2 - Environment, Drive, frozen engine loader

In [2]:
from google.colab import drive
from pathlib import Path
import time

drive.flush_and_unmount()
time.sleep(3)
drive.mount('/content/drive', force_remount=True)

root = Path('/content/drive/MyDrive/GEMS_2')
ledger = root / 'environment/sync_ledger.json'

for attempt in range(1, 5):
    try:
        print("Drive root accessible:", root.is_dir())
        print("Sync ledger exists:", ledger.is_file())
        if ledger.is_file():
            print("Ledger size:", ledger.stat().st_size, "bytes")
        print("Drive connection: OK")
        break
    except (OSError, ConnectionError) as e:
        print(f"Drive access failed ({attempt}/4): {e}")
        if attempt == 4:
            raise RuntimeError(
                "Google Drive mount is unavailable. Restart the Colab runtime "
                "and mount Drive again before proceeding."
            ) from e
        time.sleep(3 * attempt)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
Drive root accessible: True
Sync ledger exists: True
Ledger size: 956003 bytes
Drive connection: OK


In [3]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'geomoe_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]', 'pytest'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD

import pandas as pd
from IPython.display import Image, Markdown, display
import gems_bench.geomoe as GM
from gems_bench.geomoe import run as GR
from gems_bench.leaderboard_focus import factory as FA
assert GM.GEOMOE_CODE_REV == EXPECT['code_rev'], f'code rev {GM.GEOMOE_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
NB = GR.notebook_config(CONFIG)
F = FA.Factory(GR.factory_config(NB), HEAD)     # frozen-engine loader only (no factory candidate is exported)
G = GR.DiscoveryGeoMoE(CONFIG, HEAD, factory=F)
RUN_ALL = G.run                                   # optional one-call entry point == every cell below
OUT = G.ws.root
FULL = NB['mode'] != 'FAST_DIAGNOSTIC'

def show(rel, cols=None, n=40):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

print('SHA', HEAD, '| code rev', GM.GEOMOE_CODE_REV, '| mode', NB['mode'])
print('Drive outputs:', G.drive / G.out)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[sync in] 154 copied (272.7 MiB), 0 unchanged, 122s: ['outputs/discovery_geomoe/manifests', 'outputs/discovery_geomoe/audit', 'outputs/discovery_geomoe/validation', 'outputs/discovery_geomoe/candidates', 'outputs/discovery_geomoe/experts', 'outputs/discovery_geomoe/gating', 'outputs/discovery_geomoe/submissions']
SHA 2af50315011838b664d51017cc66bc697c6bd389 | code rev dgm.2-r1 | mode FULL
Drive outputs: /content/drive/MyDrive/GEMS_2/outputs/discovery_geomoe


## 3 - [00] Synthetic tests (enforced)

Runtime self-tests + `pytest tests/test_discovery_geomoe.py`. A nonzero pytest exit code or a failed critical self-test = `TESTS_FAILED`: the full output is saved to `audit/pytest_output.txt`, the failing tests are listed, and stages [04]-[12] are blocked.

In [4]:
M00 = G.s00_tests()
if G.status['00_tests'] == 'TESTS_FAILED':
    print('\nTESTS_FAILED - modeling, selection and export are blocked. Failing:')
    for w in M00.get('failing', []): print('  ', w)
    print((OUT / 'audit/pytest_output.txt').read_text()[-4000:])

      self-tests 24/24 PASS; pytest exit code 0
[sync out] 154 copied (272.7 MiB), 0 unchanged, 7s: ['outputs/discovery_geomoe']
[00] Synthetic tests ........... COMPLETE | all tests passed


## 4 - [01]-[03] Frozen artifacts, audit (+ optional G02 backfill), scenarios

In [5]:
G.s01_attach(); G.s02_audit(); G.s03_scenarios()
show('audit/component_inventory.csv')
print(json.dumps(G.ws.read_json('audit/g02_backfill.json'), indent=1, default=str))
show('validation/scenarios.csv')

      w2_champion     FAILED | RuntimeError: synthetic programming error
      w2_crest        UNAVAILABLE | synthetic: not reconstructable from frozen artifacts
      w2_skeleton     FAILED | KeyError: 'synthetic'
      b0_b1_fallback  BLOCKED | artifact x: MISSING
[self-test] 14/14 PASS
[sync in] 88 copied (2.2 MiB), 0 unchanged, 63s: ['outputs/leaderboard_focus/submission_factory']
[sync out] 89 copied (2.2 MiB), 0 unchanged, 2s: ['outputs/leaderboard_focus/submission_factory']
[sync out] 6 copied (0.0 MiB), 84 unchanged, 0s: ['outputs/leaderboard_focus/submission_factory']
[sync out] 90 copied (2.2 MiB), 0 unchanged, 2s: ['outputs/leaderboard_focus/submission_factory']
[1/6] Drive preflight ....... WARNING | 2/2 candidates unblocked; M1 preflight COMPLETE, inventory COMPLETE
[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/config_snapshot.json', 'reports/pipeline/environment_report.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/st

,component,available,executed,reusable,limitation,action
0,B0 / B1 honest CNN maps,True,yes,yes,OOF on geo folds; discovery maps on disc splits,E1 sources
1,W2 cross-fitted held-out maps,True,yes,yes,per-unit ranks only,"E1 source, A baselines"
2,W2 unit features,True,yes,yes,one PF scale,expert layers
3,G01 / G02 rasters,True,yes (FAST_RESEARCH),yes,"units with rasters ['disc_official_0', 'disc_o...",E3 (consistent-availability rule)
4,Potential fields,True,yes,yes,second scale computed here,E2
5,Cached DEM mosaic,False,pilot,cached only,never downloaded,E4 dem
6,Known-network geometry,True,yes,yes,permitted known per scenario,E4 network + tips
7,Submission factory writer / validator,True,yes,yes,zone-stitched only,stage 12
8,Official scorer (FastScorer),True,yes,yes,OFFICIAL_EXCLUDE_SUPPORTS_TP: repo reading,every score


{
 "status": "MISSING_NOT_BACKFILLED",
 "missing": [
  "geo_2",
  "disc_official_2"
 ],
 "detail": "set backfill_missing_g02=True to compute only these units with the frozen G02 config"
}


,scenario,zone,unit,task,label,base_unit,eval_px,scored_truth_px,excluded_px,known_context_px,n_systems,sources
0,V1,0,V1:0,disc_official,entire withheld fault systems (existing discov...,disc_official:0,1784107,7579,51104,51104,38,split discovery CNN + W2 held-out
1,V1,1,V1:1,disc_official,entire withheld fault systems (existing discov...,disc_official:1,1848909,7469,52246,52246,38,split discovery CNN + W2 held-out
2,V1,2,V1:2,disc_official,entire withheld fault systems (existing discov...,disc_official:2,1724912,7282,53704,53704,44,split discovery CNN + W2 held-out
3,V3,0,V3:0,geo,buffered geographic generalisation (existing g...,geo:0,1887813,18707,0,25896,158,fold-k OOF CNN + W2 held-out
4,V3,1,V3:1,geo,buffered geographic generalisation (existing g...,geo:1,1548292,22008,0,2916,241,fold-k OOF CNN + W2 held-out
5,V3,2,V3:2,geo,buffered geographic generalisation (existing g...,geo:2,1729747,20179,0,26048,217,fold-k OOF CNN + W2 held-out
6,V2,0,V2:0,v2_net,fault-network completion (real mapped sub-sect...,geo:0,1887813,4983,13724,39620,360,fold-k OOF CNN + W2 held-out
7,V2,1,V2:1,v2_net,fault-network completion (real mapped sub-sect...,geo:1,1548292,6390,15618,18534,452,fold-k OOF CNN + W2 held-out
8,V2,2,V2:2,v2_net,fault-network completion (real mapped sub-sect...,geo:2,1729747,4929,15250,41298,368,fold-k OOF CNN + W2 held-out


,scenario,zone,unit,task,label,base_unit,eval_px,scored_truth_px,excluded_px,known_context_px,n_systems,sources
0,V1,0,V1:0,disc_official,entire withheld fault systems (existing discov...,disc_official:0,1784107,7579,51104,51104,38,split discovery CNN + W2 held-out
1,V1,1,V1:1,disc_official,entire withheld fault systems (existing discov...,disc_official:1,1848909,7469,52246,52246,38,split discovery CNN + W2 held-out
2,V1,2,V1:2,disc_official,entire withheld fault systems (existing discov...,disc_official:2,1724912,7282,53704,53704,44,split discovery CNN + W2 held-out
3,V3,0,V3:0,geo,buffered geographic generalisation (existing g...,geo:0,1887813,18707,0,25896,158,fold-k OOF CNN + W2 held-out
4,V3,1,V3:1,geo,buffered geographic generalisation (existing g...,geo:1,1548292,22008,0,2916,241,fold-k OOF CNN + W2 held-out
5,V3,2,V3:2,geo,buffered geographic generalisation (existing g...,geo:2,1729747,20179,0,26048,217,fold-k OOF CNN + W2 held-out
6,V2,0,V2:0,v2_net,fault-network completion (real mapped sub-sect...,geo:0,1887813,4983,13724,39620,360,fold-k OOF CNN + W2 held-out
7,V2,1,V2:1,v2_net,fault-network completion (real mapped sub-sect...,geo:1,1548292,6390,15618,18534,452,fold-k OOF CNN + W2 held-out
8,V2,2,V2:2,v2_net,fault-network completion (real mapped sub-sect...,geo:2,1729747,4929,15250,41298,368,fold-k OOF CNN + W2 held-out


## 5 - [04] Candidate generation (validated unit cache)

Per unit: label-free layers -> pools -> inventory (saved first) -> coverage parts (saved second). A unit is reused only when its fingerprint, arrays and tables validate; only invalid parts are recomputed.

In [6]:
G.s04_candidates()
show('candidates/source_provenance.csv', n=60)

[04] Candidate generation ...... REUSED_VALID_CACHE | fingerprint b5780348b9


,unit,source,family,status,pool_px
0,V1:0,w2,E1_learned,OK,51314.0
1,V1:0,fusion,E1_learned,OK,67677.0
2,V1:0,cnn_b0,E1_learned,OK,58119.0
3,V1:0,cnn_b1,E1_learned,OK,61552.0
4,V1:0,g02c,E3_structural,OK,40256.0
5,V1:0,g02a,E3_structural,OK,30562.0
6,V1:0,g01_persistence,E3_structural,OK,63393.0
7,V1:0,mag_hgrad,E2_potential,OK,22297.0
8,V1:0,grav_hgrad,E2_potential,OK,8634.0
9,V1:0,mg_cross,E2_potential,OK,24419.0


,unit,source,family,status,pool_px
0,V1:0,w2,E1_learned,OK,51314.0
1,V1:0,fusion,E1_learned,OK,67677.0
2,V1:0,cnn_b0,E1_learned,OK,58119.0
3,V1:0,cnn_b1,E1_learned,OK,61552.0
4,V1:0,g02c,E3_structural,OK,40256.0
...,...,...,...,...,...
121,V2:2,mg_cross,E2_potential,OK,22443.0
122,V2:2,pf_persistence,E2_potential,OK,14805.0
123,V2:2,dem,E4_terrain_network,UNAVAILABLE,NaN
124,V2:2,network,E4_terrain_network,OK,10139.0


## 6 - [05] Oracle analysis + frozen submitted W2 + [06] bottleneck diagnosis

In [7]:
G.s05_oracle(); G.s06_decision()
show('validation/frozen_w2_baseline.csv', ['scenario', 'zone', 'rep', 'frac', 'area_frac_eligible', 'dti', 'policy_source'])
show('audit/baseline_policy_parity.csv')
show('candidates/oracle_coverage.csv')
show('audit/candidate_bottleneck_reconciliation.csv')
show('validation/v2_coverage_by_subtype.csv')
print(json.dumps(G.ws.read_json('candidates/coverage_decision.json'), indent=1, default=str)[:3000])

[05] Oracle analysis ........... REUSED_VALID_CACHE | fingerprint ccd3ce270a
[06] Bottleneck diagnosis ...... REUSED_VALID_CACHE | fingerprint 85b54859fa


,scenario,zone,rep,frac,area_frac_eligible,dti,policy_source
0,V1,0,skeleton_1px,0.01,0.003904,0.269027,factory selection/champion_candidates.csv row ...
1,V1,1,skeleton_1px,0.01,0.004551,0.259944,factory selection/champion_candidates.csv row ...
2,V1,2,skeleton_1px,0.01,0.005472,0.198579,factory selection/champion_candidates.csv row ...
3,V3,0,skeleton_1px,0.01,0.004053,0.128571,factory selection/champion_candidates.csv row ...
4,V3,1,skeleton_1px,0.01,0.003882,0.086911,factory selection/champion_candidates.csv row ...
5,V3,2,skeleton_1px,0.01,0.003935,0.104271,factory selection/champion_candidates.csv row ...
6,V2,0,skeleton_1px,0.01,0.004741,0.079701,factory selection/champion_candidates.csv row ...
7,V2,1,skeleton_1px,0.01,0.004387,0.064919,factory selection/champion_candidates.csv row ...
8,V2,2,skeleton_1px,0.01,0.004594,0.065409,factory selection/champion_candidates.csv row ...


,unit,scenario,rep,frac,blend,grid_candidate,area_frac_eligible,area_px,jaccard_vs_factory_evaluation_map,w2_map_sha256,recipe_id,policy_source,grid_max_abs_diff,grid_status,status
0,V1:0,V1,skeleton_1px,0.01,1.0,boost_a:w2,0.003904,6897,1.0,b069fab5b73112c82f54fa4c797d4cd9411fabe67670c2...,0a8692b2ab,factory selection/champion_candidates.csv row ...,2.273737e-13,PASS,PASS
1,V1:1,V1,skeleton_1px,0.01,1.0,boost_a:w2,0.004551,8346,1.0,5bb95cf7b3773751325d164bcf000ea45d7363c0d3815e...,0a8692b2ab,factory selection/champion_candidates.csv row ...,2.273737e-13,PASS,PASS
2,V1:2,V1,skeleton_1px,0.01,1.0,boost_a:w2,0.005472,9342,1.0,b042ad9210034966fe8f0b1983d854763b98d5fe16e564...,0a8692b2ab,factory selection/champion_candidates.csv row ...,0.000000e+00,PASS,PASS
3,V3:0,V3,skeleton_1px,0.01,1.0,boost_a:w2,0.004053,7648,1.0,6e93c29a281fb1636a0e8d47345bdb43dcc4b6a210b820...,0a8692b2ab,factory selection/champion_candidates.csv row ...,0.000000e+00,PASS,PASS
4,V3:1,V3,skeleton_1px,0.01,1.0,boost_a:w2,0.003882,6009,1.0,8c157bfb99aac31e98272299d1b5af78ed2022c85645ce...,0a8692b2ab,factory selection/champion_candidates.csv row ...,0.000000e+00,PASS,PASS
5,V3:2,V3,skeleton_1px,0.01,1.0,boost_a:w2,0.003935,6806,1.0,3b54b237cac6047d0449df69e41ebb4775fe7f16c9e71f...,0a8692b2ab,factory selection/champion_candidates.csv row ...,0.000000e+00,PASS,PASS
6,V2:0,V2,skeleton_1px,0.01,1.0,boost_a:w2,0.004741,8881,NaN,6e93c29a281fb1636a0e8d47345bdb43dcc4b6a210b820...,0a8692b2ab,factory selection/champion_candidates.csv row ...,NaN,NOT_APPLICABLE (V2 is new),UNVERIFIED
7,V2:1,V2,skeleton_1px,0.01,1.0,boost_a:w2,0.004387,6723,NaN,8c157bfb99aac31e98272299d1b5af78ed2022c85645ce...,0a8692b2ab,factory selection/champion_candidates.csv row ...,NaN,NOT_APPLICABLE (V2 is new),UNVERIFIED
8,V2:2,V2,skeleton_1px,0.01,1.0,boost_a:w2,0.004594,7875,NaN,3b54b237cac6047d0449df69e41ebb4775fe7f16c9e71f...,0a8692b2ab,factory selection/champion_candidates.csv row ...,NaN,NOT_APPLICABLE (V2 is new),UNVERIFIED


,scenario,n_systems,cov_fam:E1_learned,lencov_fam:E1_learned,cov_fam:E2_potential,lencov_fam:E2_potential,cov_fam:E3_structural,lencov_fam:E3_structural,cov_fam:E4_terrain_network,lencov_fam:E4_terrain_network,...,w2_recovered_share,unique_E1_learned,unique_E2_potential,unique_E3_structural,unique_E4_terrain_network,independent_not_E1,independent_not_W2,evaluated_E2_potential,evaluated_E3_structural,evaluated_E4_terrain_network
0,V1,120,0.850000,0.779176,0.216667,0.257053,0.266667,0.281281,0.000000,0.015181,...,0.166667,63,0,3,0,3,36,True,True,True
1,V2,1180,0.595763,0.568642,0.283898,0.277451,0.316949,0.316710,0.781356,0.707275,...,0.079661,66,15,20,214,411,959,True,True,True
2,V3,616,0.582792,0.571353,0.227273,0.259829,0.316558,0.285365,0.035714,0.038395,...,0.047078,178,20,40,2,86,252,True,True,True


,scenario,pool,definition,n_systems,system_coverage,length_coverage,missed_by_w2_systems,ranking_limited_length_share,generation_limited_length_share,tolerance_m,covered_rule,budget,status,reading
0,V1,label_free,M7-like: E2 + E3 lines only,120,0.408333,0.414510,100,0.405152,0.594848,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
1,V1,fam:E1_learned,X2-like: CNN / W2 / fusion lines,120,0.850000,0.779176,100,0.860724,0.139276,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
2,V1,inventory,unified pool (all families),120,0.775000,0.720690,100,0.782183,0.217817,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
3,V2,label_free,M7-like: E2 + E3 lines only,1180,0.463559,0.461048,1086,0.506120,0.493880,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
4,V2,fam:E1_learned,X2-like: CNN / W2 / fusion lines,1180,0.595763,0.568642,1086,0.581583,0.418417,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
5,V2,inventory,unified pool (all families),1180,0.894068,0.842841,1086,0.894103,0.105897,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
6,V3,label_free,M7-like: E2 + E3 lines only,616,0.444805,0.425444,587,0.435528,0.564472,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
7,V3,fam:E1_learned,X2-like: CNN / W2 / fusion lines,616,0.582792,0.571353,587,0.644297,0.355703,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...
8,V3,inventory,unified pool (all families),616,0.654221,0.621506,587,0.687357,0.312643,300.0,>= 0.5 of scored pixels within tolerance,0.05 of eligible per source,MEASURED,A discrepancy between rows of one scenario is ...


,subtype,n_events,truth_length_km,inventory_coverage,w2_recovered_share,coverage_E1_learned,coverage_E2_potential,coverage_E3_structural,coverage_E4_terrain_network
0,branch,258,212.016,0.891473,0.050388,0.569767,0.368217,0.356589,0.817829
1,short_splay,548,538.832,0.861314,0.083942,0.593066,0.277372,0.299270,0.697080
2,subsection,32,44.464,0.937500,0.218750,0.625000,0.312500,0.312500,1.000000
3,tip_extension,342,1030.512,0.944444,0.081871,0.616959,0.228070,0.315789,0.868421


{
 "overall": "RANKING_LIMITED",
 "per_scenario": {
  "V1": {
   "verdict": "RANKING_LIMITED",
   "pool_coverage": 0.775,
   "w2_recovered": 0.1666666666666666,
   "gap": 0.6083333333333334,
   "n_systems": 120,
   "independent_not_E1": 3,
   "independent_share": 0.025,
   "families_evaluated": {
    "E2_potential": true,
    "E3_structural": true,
    "E4_terrain_network": true
   }
  },
  "V2": {
   "verdict": "RANKING_LIMITED",
   "pool_coverage": 0.8940677966101694,
   "w2_recovered": 0.0796610169491525,
   "gap": 0.8144067796610169,
   "n_systems": 1180,
   "independent_not_E1": 411,
   "independent_share": 0.3483050847457627,
   "families_evaluated": {
    "E2_potential": true,
    "E3_structural": true,
    "E4_terrain_network": true
   }
  },
  "V3": {
   "verdict": "RANKING_LIMITED",
   "pool_coverage": 0.6542207792207793,
   "w2_recovered": 0.047077922077922,
   "gap": 0.6071428571428572,
   "n_systems": 616,
   "independent_not_E1": 86,
   "independent_share": 0.139610389610

## 7 - [07]-[10] Experts, static fusion, geological rules, learned GeoMoE (FULL only)

In [ ]:
if FULL:
    G.s07_experts(); G.s08_static(); G.s09_rules(); G.s10_learned()
    A = show('experts/source_availability.csv', n=0)
    if A is not None: display(A[A.member == '*'])
else:
    print('FAST_DIAGNOSTIC: stages [07]-[12] not run (set mode FULL to continue from the cached artifacts)')

/content/GEMS/gems_bench/geomoe/experts.py:125: RuntimeWarning: Mean of empty slice
  raw = np.nanmean(np.stack(vals), 0) if vals else np.full(len(T), np.nan)


[sync out] 8 copied (5.9 MiB), 150 unchanged, 0s: ['outputs/discovery_geomoe']
[07] Expert evidence ........... COMPLETE | cross-fitted ECDF normalisation; E3 modes ['dropped (unavailable in zones [2])']


## 8 - [10b] Model comparison and selection, [11] deployment, [12] submission (FULL only)

Fixed-policy evaluation (one rule for every scenario, development units only) vs the frozen submitted W2; paired block bootstrap; effect decomposition; V2 by subtype. Export only of the selected model (or EXPLORATORY when allowed); nothing is uploaded.

In [ ]:
if FULL:
    G.s10b_compare(); G.s11_deployment(); G.s12_export()
    show('gating/model_summary.csv', ['scenario', 'model', 'protocol', 'dti', 'worst_zone_dti', 'params_all'])
    show('gating/paired_comparison.csv'); show('gating/effect_decomposition.csv'); show('validation/v2_by_subtype.csv')
    print(json.dumps({k: G.ws.read_json('gating/selection.json').get(k) for k in ('selected', 'status', 'measured', 'reasons')}, indent=1, default=str))
    show('submissions/submission_manifest.csv')

## 9 - [14] Stage integrity, [13] reports and figures

In [ ]:
G.s14_integrity(); G.s13_reports()
show('audit/stage_integrity.csv', ['stage', 'actual_status', 'missing_outputs', 'failure_reason', 'affected_file', 'scientific_conclusion_allowed'])
for f in ('baseline_comparison', 'oracle_vs_actual', 'unique_geology_recovery', 'v2_fault_subtypes', 'gating_comparison', 'geographic_robustness'):
    p = OUT / 'figures' / f'{f}.png'
    if p.exists(): display(Image(str(p)))
for r in ('EXECUTION_STATUS', 'SCIENTIFIC_FINDINGS', 'LEADERBOARD_RECOMMENDATION', 'NEXT_ACTION'):
    display(Markdown((OUT / f'reports/{r}.md').read_text()))
G.final_message()